In [2]:
import os
import cv2
from tqdm import tqdm
import numpy as np

# --- Configuração dos Caminhos ---
DIRETORIO_IMAGENS_ORIGINAIS = "pneumonia"
DIRETORIO_MASCARAS = "pneumonia_mascaras"
DIRETORIO_SAIDA_SEGMENTADO = "pneumonia_segmented"

print(f"Criando o novo dataset em: '{DIRETORIO_SAIDA_SEGMENTADO}'")

for subset in ["train", "val", "test"]:
    print(f"\nProcessando o conjunto: {subset}...")
    
    pasta_imagens_subset = os.path.join(DIRETORIO_IMAGENS_ORIGINAIS, subset)
    pasta_mascaras_subset = os.path.join(DIRETORIO_MASCARAS, subset)
    pasta_saida_subset = os.path.join(DIRETORIO_SAIDA_SEGMENTADO, subset)

    lista_de_imagens = []
    for raiz, _, arquivos in os.walk(pasta_imagens_subset):
        for nome_arquivo in arquivos:
            if nome_arquivo.lower().endswith(('.png', '.jpg', '.jpeg')):
                lista_de_imagens.append(os.path.join(raiz, nome_arquivo))

    for caminho_imagem in tqdm(lista_de_imagens, desc=f"Aplicando máscaras em '{subset}'"):
        try:
            nome_base, extensao = os.path.splitext(os.path.basename(caminho_imagem))
            nome_mascara = f"{nome_base}_mask.png"
            subpasta_classe = os.path.basename(os.path.dirname(caminho_imagem))
            caminho_mascara = os.path.join(pasta_mascaras_subset, subpasta_classe, nome_mascara)

            if os.path.exists(caminho_mascara):
                imagem_original = cv2.imread(caminho_imagem, cv2.IMREAD_GRAYSCALE)
                mascara = cv2.imread(caminho_mascara, cv2.IMREAD_GRAYSCALE)
                
                # >>>>> INÍCIO DA CORREÇÃO <<<<<
                # Garante que a imagem original tenha o mesmo tamanho da máscara
                # Pega as dimensões da máscara (altura, largura)
                altura, largura = mascara.shape
                # Redimensiona a imagem original para essas dimensões
                imagem_original_redimensionada = cv2.resize(imagem_original, (largura, altura))
                # >>>>> FIM DA CORREÇÃO <<<<<
                
                mascara_normalizada = mascara.astype(np.float32) / 255.0
                
                # Agora a multiplicação vai funcionar
                imagem_segmentada = cv2.multiply(imagem_original_redimensionada.astype(np.float32), mascara_normalizada)
                imagem_segmentada = imagem_segmentada.astype(np.uint8)

                caminho_saida_final = os.path.join(pasta_saida_subset, subpasta_classe)
                os.makedirs(caminho_saida_final, exist_ok=True)
                
                cv2.imwrite(os.path.join(caminho_saida_final, os.path.basename(caminho_imagem)), imagem_segmentada)
            else:
                print(f"AVISO: Máscara não encontrada para a imagem {caminho_imagem}")

        except Exception as e:
            print(f"ERRO ao processar {caminho_imagem}: {e}")

print("\n✅ Processo de criação do dataset segmentado concluído!")

Criando o novo dataset em: 'pneumonia_segmented'

Processando o conjunto: train...


Aplicando máscaras em 'train': 100%|███████████████████████████████████████████████| 5216/5216 [01:25<00:00, 60.76it/s]



Processando o conjunto: val...


Aplicando máscaras em 'val': 100%|███████████████████████████████████████████████████| 640/640 [00:06<00:00, 98.53it/s]



Processando o conjunto: test...


Aplicando máscaras em 'test': 100%|████████████████████████████████████████████████| 2831/2831 [01:11<00:00, 39.73it/s]


✅ Processo de criação do dataset segmentado concluído!
